# 02 — Select features

**Depends on:** notebooks 00 and 01. **Produces:** `ranked_features.csv`,
`selected_features.json`.

## What this stage is for

Going from every column to a short, ranked, defensible feature list.

## Why not just use everything

Because these models measure distance, and every useless dimension dilutes the
distance. A feature that is 95% null contributes almost nothing but the
imputer's fill value; a feature with zero variance contributes exactly nothing;
eight columns of Gaussian noise contribute eight dimensions of noise. In a
20-dimensional space the fraud conjunction is visible. In a 200-dimensional space
where 180 dimensions are noise, it is not — this is the curse of dimensionality
in its most practical form.

## The four steps, and why each exists

1. **Classify by name.** Some exclusions cannot be made statistically. An
   identifier that happens to be an integer looks like a perfectly good feature.
   A leaked column looks like the best feature you have ever seen. These have to
   be excluded by *knowing what they are*, which means by name and by pattern.
2. **Profile.** Null rate, variance, quantiles. Cheap and mechanical.
3. **Rank.** Score the survivors. The score here is deliberately simple and
   readable; the package's version adds a domain prior on top.
4. **Cut.** To several lengths, so the sweep can treat length as a
   hyperparameter rather than a guess.

## The domain prior, stated honestly

Features about **what a customer did** (counts, recency, bursts) are ranked above
features about **what a customer is** (tenure, account breadth). This is a belief,
not a discovery: behaviour changes when an account is compromised, attributes
mostly do not. It is applied as a bounded multiplicative boost so it can shift a
ranking but not manufacture one. It is a knob you should feel free to turn off
and re-run — and if the leaderboard in notebook 04 does not change, then the
prior was never doing anything and you should say so.

In [ ]:
import json
import os
import pathlib
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Every notebook in this track reads and writes here. Nothing else is shared:
# no imports between notebooks, no global state, no kernel that has to stay
# alive. That is what "standalone" means -- you can run notebook 03 tomorrow on
# a different machine as long as this directory came with you.
RUN_DIR = pathlib.Path(os.environ.get("TS05_RUN_DIR", "./_run")).resolve()
RUN_DIR.mkdir(parents=True, exist_ok=True)

SEED = 20260601
rng = np.random.default_rng(SEED)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

print("run directory:", RUN_DIR)
print("contents     :", sorted(p.name for p in RUN_DIR.iterdir()) or "(empty)")

In [ ]:
manifest = json.loads((RUN_DIR / "manifest.json").read_text())
assert "01" in manifest["stages_complete"], "Run 01-explore-and-validate.ipynb first."

training = pd.read_parquet(manifest["paths"]["training_nonfraud"])
report = json.loads((RUN_DIR / "quality_report.json").read_text())

print("training:", training.shape)
print("quality severity:", report["severity"])

## Step 1 — classify every column

Each column gets exactly one verdict and one reason. Print the reason counts and
read them; a column you expected to be a candidate and is not is the interesting
case, and the reason tells you which rule caught it.

The patterns below are the readable equivalent of `ID_PATTERNS`, `TS_PATTERNS`
and `NUMERIC_DTYPE_PATTERN` in `trust_score_05/ml/selection.py`. One note from
that module's history: the numeric-dtype test was once a substring match, so
`bigint` matched on `int` and so did `point_in_time_string`. It is an exact
membership test now, and the same is true here — never test a dtype with `in`.

In [ ]:
import re

JOIN_KEY = manifest["join_key"]
LABEL_COLUMN = manifest["label_column"]
PROVENANCE = set(manifest["provenance_columns"]) | {"case_key", "fraud_ts", "fraud_type"}

# Leakage comes from two places: what we declared in 00, and what notebook 01
# detected statistically. Union them -- a declared list goes stale, and a
# detector alone will miss a leak that happens to be weak in this sample.
DECLARED_LEAKS = set(manifest.get("known_leakage_columns", []))
DETECTED_LEAKS = set()
for entry in report["checks"]:
    if entry["name"] == "suspected_label_leakage":
        DETECTED_LEAKS |= set(entry["details"].get("columns", []))
LEAKAGE = DECLARED_LEAKS | DETECTED_LEAKS

ID_PATTERNS = [r"_id$", r"^id_", r"_id_", r"_key$", r"_hash$", r"_numeric$", r"msisdn", r"imei"]
TS_PATTERNS = [r"_ts$", r"_date$", r"_timestamp$", r"^reference_"]

# Exact dtype names. NOT a substring test -- see the note above.
NUMERIC_DTYPES = {
    "int8", "int16", "int32", "int64",
    "uint8", "uint16", "uint32", "uint64",
    "float16", "float32", "float64",
    "Int8", "Int16", "Int32", "Int64", "Float32", "Float64",
}


def classify(column: str, dtype: str) -> tuple[bool, str]:
    if column == JOIN_KEY:
        return False, "join_key"
    if column == LABEL_COLUMN:
        return False, "label"
    if column in LEAKAGE:
        return False, "label_leakage"
    if column in PROVENANCE:
        return False, "provenance"
    if any(re.search(pattern, column) for pattern in TS_PATTERNS):
        return False, "timestamp"
    if any(re.search(pattern, column) for pattern in ID_PATTERNS):
        return False, "identifier"
    if dtype not in NUMERIC_DTYPES:
        return False, "not_numeric"
    return True, "candidate"


classification = pd.DataFrame(
    [
        {"column": column, "dtype": str(dtype), **dict(zip(["is_candidate", "reason"], classify(column, str(dtype))))}
        for column, dtype in training.dtypes.items()
    ]
)

candidates = classification.loc[classification["is_candidate"], "column"].tolist()

print(f"{len(candidates)} candidate(s) out of {len(classification)} column(s)")
print()
print(classification["reason"].value_counts().to_string())
print()
print("excluded as leakage:", sorted(LEAKAGE))
display(classification[~classification["is_candidate"]].head(20))

## Step 2 — profile

Null rate, variance, coefficient of variation and a few quantiles. Nothing
clever; the point is that everything the ranking uses is visible in this table.

In [ ]:
profile_rows = []
for column in candidates:
    series = training[column]
    values = series.dropna().astype(float)
    profile_rows.append(
        {
            "feature": column,
            "null_rate": float(series.isna().mean()),
            "n_unique": int(series.nunique(dropna=True)),
            "mean": float(values.mean()) if len(values) else np.nan,
            "std": float(values.std(ddof=1)) if len(values) > 1 else 0.0,
            "p01": float(values.quantile(0.01)) if len(values) else np.nan,
            "p50": float(values.quantile(0.50)) if len(values) else np.nan,
            "p99": float(values.quantile(0.99)) if len(values) else np.nan,
        }
    )

profile = pd.DataFrame(profile_rows)
profile["variance"] = profile["std"] ** 2
profile["cv"] = profile["std"] / profile["mean"].abs().replace(0, np.nan)

print("profiled:", len(profile))
display(profile.sort_values("null_rate", ascending=False).head(12).round(4))

## Step 3 — the hard cuts

In [ ]:
MAX_NULL_RATE = 0.90
MIN_VARIANCE = 1e-10
MIN_UNIQUE = 2

cut_reasons = {}
survivors = []
for row in profile.itertuples():
    if row.null_rate > MAX_NULL_RATE:
        cut_reasons[row.feature] = f"null_rate {row.null_rate:.2%} > {MAX_NULL_RATE:.0%}"
    elif row.variance < MIN_VARIANCE:
        cut_reasons[row.feature] = "zero variance"
    elif row.n_unique < MIN_UNIQUE:
        cut_reasons[row.feature] = f"only {row.n_unique} distinct value(s)"
    else:
        survivors.append(row.feature)

print(f"{len(survivors)} survivor(s), {len(cut_reasons)} cut")
print()
for feature, reason in sorted(cut_reasons.items()):
    print(f"  {feature:50s} {reason}")

## Step 3b — drop one of each near-duplicate pair

`cust_device_change_cnt_90d` and `cust_device_change_cnt_365d` are close to the
same column. Keeping both doubles that feature's weight in the distance without
adding information. Cut at |r| > 0.97 and keep the one with the lower null rate,
because the one with fewer nulls is the one whose values are real rather than
imputed.

In [ ]:
CORRELATION_CUTOFF = 0.97

subset = training[survivors].astype(float)
correlations = subset.corr().abs()
null_rate_of = profile.set_index("feature")["null_rate"].to_dict()

dropped = set()
pairs = []
columns = list(correlations.columns)
for i, left in enumerate(columns):
    if left in dropped:
        continue
    for right in columns[i + 1 :]:
        if right in dropped:
            continue
        value = correlations.loc[left, right]
        if np.isfinite(value) and value > CORRELATION_CUTOFF:
            loser = left if null_rate_of[left] > null_rate_of[right] else right
            dropped.add(loser)
            pairs.append({"kept": right if loser == left else left, "dropped": loser,
                          "r": round(float(value), 4)})
            if loser == left:
                break

decorrelated = [f for f in survivors if f not in dropped]
print(f"dropped {len(dropped)} near-duplicate(s); {len(decorrelated)} remain")
display(pd.DataFrame(pairs).head(20))

## Step 4 — rank, first pass, without the labels

This pass deliberately pretends we have no labels, because it is worth seeing
what that costs. Step 5 then adds them back. The score has three parts, all
visible:

- **spread** — the coefficient of variation, rank-normalised. A feature that
  varies between customers can distinguish them; one that barely varies cannot.
- **completeness** — `1 - null_rate`. A feature that is mostly imputed is mostly
  the imputer's opinion.
- **behavioural boost** — the domain prior described at the top.

Rank-normalising rather than using the raw CV matters: CV is unbounded and one
heavy-tailed feature would otherwise dominate the whole ranking on its own.

In [ ]:
BEHAVIOURAL_TOKENS = [
    "_cnt_", "_any_", "_days_since_last", "_interval_", "_entropy",
    "_over_", "_spread_days", "_new_", "_burst",
]
ATTRIBUTE_TOKENS = ["_days_since_first_seen", "_distinct_account_cnt", "_distinct_msisdn_cnt"]
BOOST = 0.25

ranked = profile[profile["feature"].isin(decorrelated)].copy()
ranked["spread"] = ranked["cv"].fillna(0.0).rank(pct=True)
ranked["completeness"] = 1.0 - ranked["null_rate"]
ranked["is_behavioural"] = ranked["feature"].apply(
    lambda name: any(token in name for token in BEHAVIOURAL_TOKENS)
    and not any(token in name for token in ATTRIBUTE_TOKENS)
)
ranked["boost"] = np.where(ranked["is_behavioural"], 1.0 + BOOST, 1.0)
ranked["score_unsupervised"] = ranked["spread"] * ranked["completeness"] * ranked["boost"]
ranked = ranked.sort_values("score_unsupervised", ascending=False).reset_index(drop=True)

print(f"{ranked['is_behavioural'].sum()} of {len(ranked)} features got the behavioural boost")
print()
print("top 15 by the unsupervised score alone:")
for name in ranked.head(15)["feature"]:
    print(f"  {name}")

## Step 5 — bring in the labels

Look hard at that list. Several of the eight pure-noise columns are near the top
of it, and most of the features that the fraud was actually planted in are not.
That is not a bug in the code; it is the honest limit of what an unsupervised
ranking can do. Coefficient of variation rewards a column for *varying*, and
Gaussian noise varies beautifully. So does a 3-day count column that is zero for
99% of customers and one for the rest. Neither one knows anything about fraud.

If you stop here, the models train on noise and score at a lift of about 1.0,
and the notebook reads as though isolation forests do not work. They work fine.
Selection threw the evidence away before training ever started.

The cure is sitting in the run directory. We have labels — a real, confirmed
fraud population — and notebook 01 already measured how far each feature's fraud
mean sits from its non-fraud mean. Declining to use that is not caution, it is
just waste. Three rules keep it honest:

1. **The leak check runs first.** Anything with an implausible effect size was
   already removed in step 1, so a label-derived column cannot be promoted by
   this step; it is not in the pool to promote.
2. **Labels rank features, they never become one.** No column computed from
   `label` enters the model matrix. The blend below touches the ordering only.
3. **Only training fraud is used.** The effect sizes come from
   `training_nonfraud` versus the fraud frame. The testing split is untouched,
   so the metrics in 04 are still measured on data that selection never saw.

The effect size goes through a saturating transform, `d / (d + 0.5)`, rather
than in raw. Raw would let one enormous feature flatten everything else, and
rank-normalising would do the opposite — squash a genuine 3.5 and an accidental
0.2 into neighbouring percentiles. Saturating keeps the gap where it matters
(0.2 -> 0.29, 2.0 -> 0.80, 3.5 -> 0.88) and stops caring past the point where
more separation tells you nothing new.

In [ ]:
effects = pd.read_csv(RUN_DIR / "univariate_effects.csv")
effect_lookup = dict(zip(effects["feature"], effects["abs_effect"]))

# A feature with no measured effect gets 0.0, not the mean. If notebook 01 could
# not measure it -- too few non-null fraud rows, zero pooled variance -- then we
# have no evidence for it, and "no evidence" should not be rewarded.
ranked["abs_effect"] = ranked["feature"].map(effect_lookup).fillna(0.0)
ranked["supervised"] = ranked["abs_effect"] / (ranked["abs_effect"] + 0.5)

# The unsupervised part is kept as a floored multiplier rather than dropped. It
# is what stops the ranking from promoting a feature that separates the classes
# on paper but is 80% null, or constant, or one of five copies of itself.
ranked["score"] = (
    (0.5 + ranked["supervised"])
    * (0.5 + 0.5 * ranked["spread"])
    * ranked["completeness"]
    * ranked["boost"]
)
ranked = ranked.sort_values("score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

display(
    ranked.head(30)[
        ["rank", "feature", "score", "supervised", "abs_effect", "spread",
         "completeness", "is_behavioural"]
    ].round(4)
)

# The check that the blend did its job: the columns the fraud was planted in
# should now outrank pure noise. On real data replace this with the features
# your investigators would name.
noise_best = ranked[ranked["feature"].str.startswith("cust_noise_")]["rank"].min()
print()
print(f"best-ranked pure-noise column: rank {noise_best} of {len(ranked)}")
print("features above it:", ranked[ranked["rank"] < noise_best]["feature"].tolist())

## Did the traps get caught?

This is the cell that validates the whole notebook. On the synthetic data:
`fraud_case_score` must be absent (leakage), `customer_id_numeric` must be absent
(identifier), `cust_constant_cnt` must be absent (variance), `cust_sparse_flag`
must be absent (null rate), and the eight noise columns should be in the bottom
half. If any of that fails, the selection logic is wrong — fix it here before
training anything.

In [ ]:
final = set(ranked["feature"])
traps = {
    "fraud_case_score": "label leakage",
    "customer_id_numeric": "identifier",
    "cust_constant_cnt": "zero variance",
    "cust_sparse_flag": "95% null",
}

ok = True
for name, why in traps.items():
    if name not in training.columns:
        print(f"  n/a      {name} (not in this dataset)")
        continue
    caught = name not in final
    ok &= caught
    print(f"  {'PASS' if caught else 'FAIL':8s} {name} -- should be excluded: {why}")

noise = ranked[ranked["feature"].str.startswith("cust_noise_")]
if len(noise):
    median_rank = len(ranked) / 2
    print()
    print(f"noise features rank {sorted(noise['rank'].tolist())} of {len(ranked)}")
    print(f"  {'PASS' if noise['rank'].median() > median_rank else 'WEAK':8s} "
          f"noise median rank {noise['rank'].median():.0f} vs midpoint {median_rank:.0f}")

assert ok, "A trap survived selection. Do not train until this passes."

## Cut to several lengths

In [ ]:
TOP_N_VALUES = [15, 25, 40, 60]

selected = {}
for n in TOP_N_VALUES:
    if n > len(ranked):
        print(f"top {n:3d}: only {len(ranked)} features available -- skipped")
        continue
    selected[str(n)] = ranked.head(n)["feature"].tolist()
    print(f"top {n:3d}: {len(selected[str(n)])} feature(s)")

print()
print("the top 15:")
for name in selected[str(TOP_N_VALUES[0])]:
    print(" ", name)

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(ranked["rank"], ranked["score"], marker=".", linewidth=1)
for n in selected:
    axes[0].axvline(int(n), linestyle="--", linewidth=0.8, alpha=0.6)
axes[0].set_title("ranking score by rank (cut points dashed)")
axes[0].set_xlabel("rank")
axes[0].set_ylabel("score")

top = ranked.head(20).iloc[::-1]
axes[1].barh(top["feature"], top["score"])
axes[1].set_title("top 20 features")
axes[1].tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.show()

In [ ]:
ranked.to_csv(RUN_DIR / "ranked_features.csv", index=False)
classification.to_csv(RUN_DIR / "column_classification.csv", index=False)
(RUN_DIR / "selected_features.json").write_text(
    json.dumps(
        {
            "notebook": "02-select-features",
            "max_null_rate": MAX_NULL_RATE,
            "min_variance": MIN_VARIANCE,
            "correlation_cutoff": CORRELATION_CUTOFF,
            "behavioural_boost": BOOST,
            "excluded_as_leakage": sorted(LEAKAGE),
            "selected": selected,
        },
        indent=2,
    )
)

manifest["stages_complete"] = sorted(set(manifest["stages_complete"]) | {"02"})
manifest["paths"].update(
    {
        "ranked_features": str(RUN_DIR / "ranked_features.csv"),
        "selected_features": str(RUN_DIR / "selected_features.json"),
        "column_classification": str(RUN_DIR / "column_classification.csv"),
    }
)
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

print("Stage 02 complete. Open 03-train-and-score.ipynb.")